# 🏥 What Makes an AI Model Good or Bad?
## An Interactive Demo for Radiotherapy Clinicians

---

Welcome to this hands-on demonstration! We are going to explore **why the quality of data used to train an AI model is crucial** — and what happens when that data is imperfect.

We will use a simple example: **handwritten digit recognition** (recognising numbers 0–9 from small images). While this sounds unrelated to radiotherapy, the exact same principles apply to the AI models you use every day:

- Models that **auto-segment** organs or tumours on CT scans
- Models that **predict** treatment outcomes or patient selection

### 🎯 Learning goals
After this demo you will understand that model quality depends on:

| # | Data property | Radiotherapy analogy |
|---|---------------|----------------------|
| 1 | **Completeness** — was the full range of cases in the training data? | Model trained on proton patients, applied to photon patients |
| 2 | **Correctness** — were the training labels accurate? | Contours drawn with inconsistent or wrong guidelines |
| 3 | **Quality** — were the training images clean? | Low-quality CT scans, different scanner protocols |
| 4 | **Volume** — was there enough training data? | Rare tumour type with only a few patients available |

---
> ℹ️ **You don't need to understand the code.** Just press ▶ **Run** on each cell (or use *Run All* from the menu).


## ⚙️ Setup
*Run this cell first to load all required tools.*

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, json, os

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

# ── Reproduce the exact same train/test split as used during pre-training ──
SEED = 42
SAVE_DIR = "models"

digits = load_digits()
X_all = digits.images / 16.0
y_all = digits.target
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all
)

# ── CNN architecture (must match pretrain_models.py) ──────────────────────
class DigitCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(2), nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

def load_model(filename, num_classes=10):
    m = DigitCNN(num_classes)
    m.load_state_dict(torch.load(f"{SAVE_DIR}/{filename}", map_location="cpu"))
    m.eval()
    return m

def predict(model, X):
    Xt = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    with torch.no_grad():
        return model(Xt).argmax(dim=1).numpy()

def accuracy(preds, labels):
    return np.mean(preds == labels)

def per_class_accuracy(preds, labels, num_classes=10):
    return {i: np.mean(preds[labels == i] == i) if np.sum(labels == i) > 0 else np.nan
            for i in range(num_classes)}

print("✅ Setup complete! All tools loaded.")
print(f"   Dataset: {len(X_train_all)} training images, {len(X_test)} test images")
print(f"   Image size: 8×8 pixels, 10 digit classes (0–9)")


---
## 🔍 First: What Does Our Data Look Like?

Let's take a look at the images we're working with. Each image is a tiny 8×8 pixel photograph of a handwritten digit.


In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(14, 6))
fig.suptitle("Sample images from the dataset — one column per digit (0–9)", 
             fontsize=13, fontweight="bold", y=1.01)

for digit in range(10):
    samples = X_train_all[y_train_all == digit][:4]
    for row, img in enumerate(samples):
        ax = axes[row, digit]
        ax.imshow(img, cmap="gray_r", interpolation="nearest")
        ax.axis("off")
        if row == 0:
            ax.set_title(str(digit), fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()
print("Each column shows 4 different handwritten versions of the same digit.")
print("The AI must learn to recognise all of these as the same number.")


---
## 📦 Scenario 1: Data Completeness
### *Was the full range of cases included in the training data?*

Imagine a hospital trained an auto-segmentation AI **only on proton therapy patients**, but then started using it for **photon therapy patients** too. 
The model has never seen that type of patient before!

In this demo: we train a model **only on digits 0–8**, then ask it to also classify digit **9**.


In [ ]:
# Load models trained with complete vs incomplete data
model_incomplete = load_model("model_incomplete.pt", num_classes=9)  # trained on 0-8 only
model_complete   = load_model("model_complete.pt",   num_classes=10) # trained on 0-9

# The 'incomplete' model can only output predictions 0-8 (it has no class for 9)
preds_incomplete = predict(model_incomplete, X_test)  # outputs 0-8 only
preds_complete   = predict(model_complete,   X_test)

# Overall accuracy
acc_inc = accuracy(preds_incomplete, y_test)
acc_com = accuracy(preds_complete,   y_test)

print(f"Overall accuracy — Incomplete model (0–8 only): {acc_inc*100:.1f}%")
print(f"Overall accuracy — Complete model   (0–9):     {acc_com*100:.1f}%")
print()

# Per-class accuracy
pca_inc = per_class_accuracy(preds_incomplete, y_test)
pca_com = per_class_accuracy(preds_complete,   y_test)

# ── Plot per-class accuracy ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Scenario 1: Data Completeness — Per-digit Accuracy", 
             fontsize=13, fontweight="bold")

for ax, pca, title, color in zip(
    axes,
    [pca_inc, pca_com],
    ["❌ Incomplete model\n(trained on 0–8 only)", "✅ Complete model\n(trained on 0–9)"],
    ["#e74c3c", "#2ecc71"]
):
    digits_shown = list(range(10))
    accs = [pca.get(d, 0) * 100 for d in digits_shown]
    bars = ax.bar(digits_shown, accs, color=[
        "#e74c3c" if (d == 9 and "Incomplete" in title) else color 
        for d in digits_shown
    ], edgecolor="white", linewidth=1.5)
    ax.set_xlabel("Digit", fontsize=12)
    ax.set_ylabel("Accuracy (%)", fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(10))
    ax.set_ylim(0, 110)
    ax.axhline(100, color="gray", linestyle="--", alpha=0.4)
    for bar, acc in zip(bars, accs):
        if acc > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                    f"{acc:.0f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Show what the incomplete model PREDICTS for digit 9
idx_9 = np.where(y_test == 9)[0][:10]
preds_for_9 = preds_incomplete[idx_9]

fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
fig.suptitle("What does the INCOMPLETE model predict when it sees a '9'?", 
             fontsize=13, fontweight="bold")

for col, (img_idx, pred) in enumerate(zip(idx_9, preds_for_9)):
    ax_top = axes[0, col]
    ax_bot = axes[1, col]
    ax_top.imshow(X_test[img_idx], cmap="gray_r", interpolation="nearest")
    ax_top.axis("off")
    ax_top.set_title("True: 9", fontsize=9)
    ax_bot.text(0.5, 0.5, f"Predicted:\n{pred}", 
                ha="center", va="center", fontsize=12, fontweight="bold",
                color="#e74c3c", transform=ax_bot.transAxes)
    ax_bot.axis("off")

plt.tight_layout()
plt.show()

print("🔑 Key insight:")
print("   The incomplete model has NEVER seen a '9' during training.")
print("   It forces every image into one of the classes it knows (0–8),")
print("   confidently giving a WRONG answer every single time.")
print()
print("   In radiotherapy: a model trained only on proton patients may")
print("   confidently produce wrong segmentations for photon patients.")


---
## ✏️ Scenario 2: Data Correctness
### *Were the training labels accurate?*

AI models learn *from the labels given to training data*. If those labels are wrong, the model learns the wrong thing.

In radiotherapy: if an AI was trained on contours that were drawn inconsistently — perhaps one institution draws the parotid gland differently from another — the model absorbs that inconsistency.

In this demo: we **randomly flip 20% of the training labels** (tell the model that some "3"s are actually "7"s, etc.) and see how performance suffers.


In [ ]:
model_correct_labels = load_model("model_complete.pt",   num_classes=10)
model_noisy_labels   = load_model("model_noisy_labels.pt", num_classes=10)

preds_correct = predict(model_correct_labels, X_test)
preds_noisy   = predict(model_noisy_labels,   X_test)

print(f"Accuracy — Correct labels:          {accuracy(preds_correct, y_test)*100:.1f}%")
print(f"Accuracy — 20% wrong labels:        {accuracy(preds_noisy,   y_test)*100:.1f}%")
print()

# Per-class accuracy
pca_correct = per_class_accuracy(preds_correct, y_test)
pca_noisy   = per_class_accuracy(preds_noisy,   y_test)

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("Scenario 2: Data Correctness — Per-digit Accuracy", 
             fontsize=13, fontweight="bold")

x = np.arange(10)
w = 0.35
bars1 = ax.bar(x - w/2, [pca_correct[i]*100 for i in range(10)], w,
               label="✅ Correct labels", color="#2ecc71", edgecolor="white")
bars2 = ax.bar(x + w/2, [pca_noisy[i]*100   for i in range(10)], w,
               label="❌ 20% wrong labels", color="#e74c3c", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels([str(i) for i in range(10)])
ax.set_xlabel("Digit", fontsize=12); ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_ylim(0, 115); ax.legend(fontsize=11)
ax.axhline(100, color="gray", linestyle="--", alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# Show examples of wrongly-labelled training images
noisy_idx = np.load(f"{SAVE_DIR}/noisy_label_indices.npy")
y_train_noisy = np.load(f"{SAVE_DIR}/y_train_noisy.npy")

# Pick 10 examples where the label was corrupted
show_idx = noisy_idx[:10]

fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
fig.suptitle("Examples of training images with WRONG labels (what the model was taught)", 
             fontsize=12, fontweight="bold")

for col, i in enumerate(show_idx):
    ax_img = axes[0, col]
    ax_lbl = axes[1, col]
    ax_img.imshow(X_train_all[i], cmap="gray_r", interpolation="nearest")
    ax_img.axis("off")
    ax_img.set_title(f"True: {y_train_all[i]}", fontsize=9, color="green")
    ax_lbl.text(0.5, 0.5, f"Taught\nas: {y_train_noisy[i]}", 
                ha="center", va="center", fontsize=10, fontweight="bold",
                color="#e74c3c", transform=ax_lbl.transAxes)
    ax_lbl.axis("off")

plt.tight_layout()
plt.show()

print("🔑 Key insight:")
print("   The model was repeatedly told '3 is a 7', '1 is a 4', etc.")
print("   Over time it builds up a confused internal representation.")
print()
print("   In radiotherapy: if different physicians drew contours differently,")
print("   the model learns an 'average' that may not be correct for anyone.")
print("   This is called INTER-OBSERVER VARIABILITY — and it's a real problem.")


---
## 📡 Scenario 3: Data Quality
### *Were the training images clean and clear?*

Not all medical images are of equal quality. A CT scan taken on an old scanner with a non-standard protocol may look quite different from one taken on a modern machine. If the AI was trained only on high-quality images, it may fail on lower-quality inputs — and vice versa.

In this demo: we **add random noise** to the training images (like static on a TV) and observe the effect on performance.


In [ ]:
X_train_noisy = np.load(f"{SAVE_DIR}/X_train_noisy.npy")

model_clean_images = load_model("model_complete.pt",    num_classes=10)
model_noisy_images = load_model("model_noisy_images.pt", num_classes=10)

preds_clean = predict(model_clean_images, X_test)
preds_noisy = predict(model_noisy_images, X_test)

print(f"Accuracy — Model trained on clean images: {accuracy(preds_clean, y_test)*100:.1f}%")
print(f"Accuracy — Model trained on noisy images: {accuracy(preds_noisy, y_test)*100:.1f}%")
print()

# Visualise: clean vs noisy training images
fig, axes = plt.subplots(3, 10, figsize=(14, 5))
fig.suptitle("Scenario 3: Data Quality — Clean vs. Noisy Training Images", 
             fontsize=13, fontweight="bold")

for col in range(10):
    idx = np.where(y_train_all == col)[0][0]
    axes[0, col].imshow(X_train_all[idx], cmap="gray_r", interpolation="nearest")
    axes[0, col].set_title(f"Digit {col}", fontsize=9)
    axes[0, col].axis("off")
    
    axes[1, col].imshow(X_train_noisy[idx], cmap="gray_r", interpolation="nearest")
    axes[1, col].axis("off")

    # Difference
    diff = np.abs(X_train_all[idx] - X_train_noisy[idx])
    axes[2, col].imshow(diff, cmap="hot", interpolation="nearest")
    axes[2, col].axis("off")

axes[0, 0].set_ylabel("Clean", fontsize=10, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("Noisy", fontsize=10, rotation=0, labelpad=40)
axes[2, 0].set_ylabel("Difference", fontsize=10, rotation=0, labelpad=40)

plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side per-class accuracy
pca_clean = per_class_accuracy(preds_clean, y_test)
pca_noisy_img = per_class_accuracy(preds_noisy, y_test)

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("Scenario 3: Data Quality — Per-digit Accuracy", fontsize=13, fontweight="bold")
x = np.arange(10); w = 0.35
ax.bar(x - w/2, [pca_clean[i]*100    for i in range(10)], w,
       label="✅ Clean images", color="#3498db", edgecolor="white")
ax.bar(x + w/2, [pca_noisy_img[i]*100 for i in range(10)], w,
       label="❌ Noisy images", color="#e67e22", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels([str(i) for i in range(10)])
ax.set_xlabel("Digit", fontsize=12); ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_ylim(0, 115); ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

print("🔑 Key insight:")
print("   A model trained on noisy images learns to recognise 'smudged' digits.")
print("   When it sees clean test images, it may still perform acceptably —")
print("   but accuracy drops, especially for digits that look similar (e.g. 3 vs 8).")
print()
print("   In radiotherapy: a model trained on a specific CT scanner protocol")
print("   may perform poorly on images from a different scanner or institution.")
print("   This is called a DOMAIN SHIFT.")


---
## 📊 Scenario 4: Data Volume
### *Was there enough training data?*

Deep learning models need to see *many* examples to learn robust patterns. When the training set is small, the model may learn the specific training examples by heart — a problem called **overfitting** — but fail to generalise to new patients.

This is a critical challenge in radiotherapy AI, because rare cancer types may have very few patients available for training.

In this demo: we compare models trained on **50**, **300**, and **~1350** images.


In [ ]:
model_tiny   = load_model("model_tiny.pt",   num_classes=10)
model_medium = load_model("model_medium.pt", num_classes=10)
model_full   = load_model("model_full.pt",   num_classes=10)

preds_tiny   = predict(model_tiny,   X_test)
preds_medium = predict(model_medium, X_test)
preds_full   = predict(model_full,   X_test)

sizes  = [50, 300, len(X_train_all)]
accs   = [accuracy(p, y_test)*100 for p in [preds_tiny, preds_medium, preds_full]]
colors = ["#e74c3c", "#e67e22", "#2ecc71"]
labels = ["Tiny\n(50 samples)", "Medium\n(300 samples)", f"Full\n({len(X_train_all)} samples)"]

for size, acc in zip(sizes, accs):
    print(f"  Training size {size:5d}  →  Test accuracy: {acc:.1f}%")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Scenario 4: Data Volume", fontsize=13, fontweight="bold")

# Bar chart
ax = axes[0]
bars = ax.bar(labels, accs, color=colors, edgecolor="white", linewidth=1.5, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha="center", va="bottom", fontsize=13, fontweight="bold")
ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_ylim(0, 110)
ax.set_title("Overall accuracy vs. training set size", fontsize=12)

# Learning curve (accuracy vs log training size)
ax2 = axes[1]
ax2.semilogx(sizes, accs, "o-", color="#2c3e50", linewidth=2.5, markersize=10)
for s, a, c in zip(sizes, accs, colors):
    ax2.scatter(s, a, color=c, s=150, zorder=5)
    ax2.annotate(f"{a:.1f}%", (s, a), textcoords="offset points", 
                 xytext=(8, 5), fontsize=11)
ax2.set_xlabel("Training set size (log scale)", fontsize=12)
ax2.set_ylabel("Test Accuracy (%)", fontsize=12)
ax2.set_ylim(0, 110)
ax2.set_title("Learning curve: more data → better performance", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Show per-class accuracy for tiny vs full model to see which digits suffer
pca_tiny = per_class_accuracy(preds_tiny, y_test)
pca_full = per_class_accuracy(preds_full, y_test)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(10); w = 0.35
ax.bar(x - w/2, [pca_tiny[i]*100 for i in range(10)], w,
       label=f"❌ Tiny (50 samples)", color="#e74c3c", edgecolor="white")
ax.bar(x + w/2, [pca_full[i]*100 for i in range(10)], w,
       label=f"✅ Full ({len(X_train_all)} samples)", color="#2ecc71", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels([str(i) for i in range(10)])
ax.set_xlabel("Digit", fontsize=12); ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Per-digit accuracy: 50 samples vs full dataset", fontsize=13, fontweight="bold")
ax.set_ylim(0, 115); ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

print("🔑 Key insight:")
print("   With only 50 training examples, the model gets very few examples")
print("   of each digit class (~5 per digit!). It cannot learn robust patterns.")
print()
print("   In radiotherapy: a model for a rare tumour type trained on 10 patients")
print("   may perform well on those 10 patients but fail on new ones.")
print("   Transfer learning and data augmentation can help, but are not magic.")


---
## 🏁 Summary: The Four Pillars of Training Data Quality


In [ ]:
# Final summary comparison across all scenarios
fig, ax = plt.subplots(figsize=(13, 6))

scenarios = [
    "Complete\ndata\n(baseline)",
    "Incomplete\ndata\n(missing digit 9)",
    "Wrong\nlabels\n(20% noise)",
    "Noisy\nimages\n(σ=0.4)",
    "Tiny\ndataset\n(50 samples)",
]

# Recompute all accuracies for clean comparison
model_baseline = load_model("model_complete.pt", num_classes=10)
model_inc      = load_model("model_incomplete.pt", num_classes=9)
model_noisy_l  = load_model("model_noisy_labels.pt", num_classes=10)
model_noisy_i  = load_model("model_noisy_images.pt", num_classes=10)
model_tiny_    = load_model("model_tiny.pt", num_classes=10)

all_accs = [
    accuracy(predict(model_baseline, X_test), y_test) * 100,
    accuracy(predict(model_inc,      X_test), y_test) * 100,
    accuracy(predict(model_noisy_l,  X_test), y_test) * 100,
    accuracy(predict(model_noisy_i,  X_test), y_test) * 100,
    accuracy(predict(model_tiny_,    X_test), y_test) * 100,
]
scenario_colors = ["#2ecc71", "#e74c3c", "#e74c3c", "#e67e22", "#e74c3c"]
issues = ["-", "Completeness", "Correctness", "Quality", "Volume"]

bars = ax.bar(scenarios, all_accs, color=scenario_colors, edgecolor="white", linewidth=1.5, width=0.55)
for bar, acc, issue in zip(bars, all_accs, issues):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha="center", va="bottom", fontsize=13, fontweight="bold")
    if issue != "-  ":
        ax.text(bar.get_x() + bar.get_width()/2, 4,
                f"↓ {issue}", ha="center", va="bottom", fontsize=9,
                color="white", fontweight="bold")

ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_ylim(0, 112)
ax.set_title("Summary: How data quality issues affect model performance", 
             fontsize=13, fontweight="bold")
ax.axhline(all_accs[0], color="#27ae60", linestyle="--", alpha=0.5, label="Baseline (complete, correct, clean, sufficient data)")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  ✅ Baseline (good data on all fronts):  {all_accs[0]:.1f}% accuracy")
print(f"  ❌ Incomplete data (missing classes):    {all_accs[1]:.1f}% accuracy")
print(f"  ❌ Wrong labels (20% label noise):       {all_accs[2]:.1f}% accuracy")
print(f"  ⚠️  Noisy images (degraded quality):     {all_accs[3]:.1f}% accuracy")
print(f"  ❌ Too little data (50 samples only):    {all_accs[4]:.1f}% accuracy")
print()
print("  These same issues affect AI models in radiotherapy.")
print("  When evaluating an AI model, always ask:")
print("  1. Was it trained on data representative of MY patients?")
print("  2. Were the training labels (contours/outcomes) reliable?")
print("  3. Were the training images of similar quality to mine?")
print("  4. Was the training dataset large enough?")


---
## 💡 Discussion Questions

Think about AI models used in your clinical workflow and discuss with your colleagues:

1. **Completeness**: Are there patient subgroups in your clinic that might not have been well represented in the training data of a commercial AI model? (e.g. paediatric patients, rare tumour types, specific treatment techniques)

2. **Correctness**: How were the training contours for auto-segmentation models created? Were they drawn by experts following a specific guideline? What happens when different institutions use different guidelines?

3. **Quality**: If a model was trained on data from one CT scanner manufacturer, what might happen when it is applied to a scanner from a different manufacturer? How could you check for this?

4. **Volume**: For a model predicting proton therapy outcomes — how many patients do you think were in the training set? What would you consider "enough"?

---
*This notebook was created for the radiotherapy AI education module — Block 2: Understanding AI Inputs, Models & Outputs.*
